# Open Insect ID v2 — Kaggle pipeline
#
# This notebook is the runnable, on-Kaggle entry point. It installs the
# `src` package from GitHub and exercises the full data pipeline. Set the
# accelerator to **CPU** for the data stages (GPU is not needed) and use the
# training notebook (02) with a **GPU**.
#
# Fill in `GITHUB_REPO` below with your repository URL (or use the local copy
# already in the Kaggle notebook's working directory).


## 1. Setup & configuration

In [ ]:
import os, subprocess, sys
from pathlib import Path

# --- Configure this to your repo -------------------------------------------
GITHUB_REPO = os.environ.get("OPENINSECT_REPO", "https://github.com/YoannDev90/OpenInsectID.git")
# ---------------------------------------------------------------------------
IS_KAGGLE = os.path.exists("/kaggle/input")

def _is_our_checkout(path):
    # Identify OUR repo precisely: any folder happening to contain a src/
    # (cwd leftovers, /kaggle, ...) must not be mistaken for the checkout.
    return (os.path.isfile(os.path.join(path, "pyproject.toml"))
            and os.path.isdir(os.path.join(path, "src", "data_pipeline")))

# Walk up from cwd (kernels start in /kaggle/working, Jupyter in the notebook
# folder) to the repo root; anywhere else means: install from GitHub.
repo_root = os.getcwd()
while not _is_our_checkout(repo_root):
    parent = os.path.dirname(repo_root)
    if parent == repo_root:
        break
    repo_root = parent
in_checkout = _is_our_checkout(repo_root)

if in_checkout:
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
else:
    # pip needs the `git+` prefix to clone -- a plain https URL makes GitHub
    # serve the HTML page and pip fails with "cannot detect archive format".
    install_url = GITHUB_REPO
    if install_url.startswith(("http://", "https://")) and not install_url.startswith("git+"):
        install_url = "git+" + install_url
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_url])

try:
    import src.data_pipeline  # noqa: F401 - fail fast, with context, if shadowed
except ModuleNotFoundError:
    print("DIAG sys.path:", sys.path)
    raise

from pathlib import Path
import yaml

print(f"cwd={os.getcwd()} checkout={in_checkout} repo_root={repo_root}")
print("package ready; importable as `src.*`")


In [ ]:
# Load the data pipeline configuration (filters, H3 resolution, curation).
from src.data_pipeline import GBIFConfig, H3Config, CurationConfig, DownloadConfig

gbif_cfg = GBIFConfig(
    class_key=216,               # Insecta
    media_type="StillImage",
    occurrence_status="PRESENT",
    basis_of_record=("HUMAN_OBSERVATION", "PRESERVED_SPECIMEN"),
    has_coordinate=True,
)
h3_cfg = H3Config(resolution=4, min_occurrences_per_cell=3)
cur_cfg = CurationConfig(min_resolution=224, max_aspect_ratio=3.0)
dl_cfg = DownloadConfig(concurrency=8, per_domain_rps=4.0, raw_root="data/raw")
print("Insecta, world-scale, H3 res", h3_cfg.resolution)


## 2. Ingest occurrences (GBIF)

Two interchangeable sources:
- **A. Async API** (`GBIFExtractor`) — best for a targeted slice / prototyping.
- **B. DuckDB over a bulk TSV** (`SpatialProcessor.load_occurrences`) — best for
  the full world-scale dump (tens of millions of rows) streamed off disk.


In [ ]:
# --- Option A: async GBIF occurrence search (prototype / bounded slice) ------
from src.data_pipeline import GBIFExtractor

extractor = GBIFExtractor(gbif_cfg)
# Limit records for a quick pass; drop `max_records=None` to page the full set.
records = extractor.extract_sync(max_records=500)
print(f"fetched {len(records)} image-bearing occurrences")
for r in records[:3]:
    print(" ", r.lineage.order, "/", r.lineage.family, "/", r.lineage.genus, "/", r.lineage.species)


In [ ]:
# --- Option B: spatial clean + H3 range mask (world-scale TSV) --------------
# Point `tsv_path` at a GBIF occurrence dump (TSV/CSV) if you have one.
from src.data_pipeline import SpatialProcessor

# sp = SpatialProcessor(h3_cfg)
# occ = sp.load_occurrences("data/gbif_insecta.tsv")   # streams via DuckDB
# clean, stats = sp.clean(occ)                          # builds H3 mask + flags outliers
# print(stats)
print("Uncomment to run DuckDB spatial cleaning on a full GBIF dump.")


## 3. Download images (async, fault-tolerant)

In [ ]:
from src.data_pipeline import ImageDownloader

downloader = ImageDownloader(dl_cfg)
# Small test slice: only the first SMALL_N records are downloaded.
# Raise it (or use records directly) for a real run.
SMALL_N = 10
result = downloader.download_sync(records[:SMALL_N])
print(f"from {SMALL_N} records -> ok={result.ok} failed={result.failed} skipped(dup)={result.skipped}")


## 4. Curation (integrity / resolution / aspect ratio)

In [ ]:
from src.data_pipeline import curate_directory

verdicts = curate_directory("data/raw", cur_cfg)
ok = sum(v.ok for v in verdicts)
print(f"{ok}/{len(verdicts)} images pass curation")
for v in verdicts:
    if not v.ok:
        print("  drop:", v.path, "->", v.reason)


## 5. Export the curated manifest

Write a Parquet manifest consumed by the training notebook. Schema:
`image_path, order, family, genus, species` (+ optional `h3_cell, lat, lon`).
Lineage is re-derived from the curated file tree. On Kaggle, push this as a
Dataset so notebook 02 can attach it as input.


In [ ]:
import polars as pl
from pathlib import Path

RAW = Path("data/raw")
schema = ["image_path", "order", "family", "genus", "species"]
rows = []
for v in verdicts:
    if not v.ok:
        continue
    parts = Path(v.path).relative_to(RAW).parts  # order/family/genus/species/hash.jpg
    if len(parts) == 5:
        o, f, g, s = parts[:4]
        rows.append({"image_path": "/".join(parts), "order": o,
                     "family": f, "genus": g, "species": s})

manifest = pl.DataFrame(rows, schema={c: pl.Utf8 for c in schema})
manifest.write_parquet("insectid_manifest.parquet")
print(f"manifest: {manifest.height} curated rows -> insectid_manifest.parquet")
# If running on Kaggle, publish manifest + images into the curated dataset
# (folder must hold dataset-metadata.json with id yoanndev90/open-insect-id-v2):
#   !kaggle datasets version -m "curated run" -p <dataset-folder> --dir-mode tar
# notebook 02 already attaches that dataset as an input.
manifest.head(5)
